# Ecommerce AWS Pipeline — Analytics Dashboard
Queries the curated S3 data lake via Athena to visualize pipeline output.

**Data source:** Brazilian Olist e-commerce dataset  
**Prerequisites:** Deploy the stack and run the generator for at least 5 minutes.

In [ ]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json, time, os

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── Config ────────────────────────────────────────────────────
with open('../cdk-outputs.json') as f:
    outputs = json.load(f)

REGION          = os.environ.get('AWS_REGION', 'us-east-1')
CURATED_BUCKET  = outputs['EcomStorageStack']['CuratedBucketName']
WORKGROUP       = 'ecom-analytics'
DATABASE        = 'ecom_datalake'

athena = boto3.client('athena', region_name=REGION)
print(f'Curated bucket: {CURATED_BUCKET}')

In [ ]:
def run_query(sql: str, max_wait: int = 60) -> pd.DataFrame:
    """Run an Athena query and return results as a DataFrame."""
    resp = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': DATABASE},
        WorkGroup=WORKGROUP,
    )
    qid = resp['QueryExecutionId']

    for _ in range(max_wait):
        status = athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status']
        state = status['State']
        if state == 'SUCCEEDED':
            break
        if state in ('FAILED', 'CANCELLED'):
            raise RuntimeError(f"Query {state}: {status.get('StateChangeReason')}")
        time.sleep(1)

    pages = athena.get_paginator('get_query_results').paginate(QueryExecutionId=qid)
    rows = []
    headers = None
    for page in pages:
        result_rows = page['ResultSet']['Rows']
        if headers is None:
            headers = [c['VarCharValue'] for c in result_rows[0]['Data']]
            result_rows = result_rows[1:]
        rows.extend([[c.get('VarCharValue', '') for c in r['Data']] for r in result_rows])

    return pd.DataFrame(rows, columns=headers)

## 1. Revenue Overview

In [ ]:
kpis = run_query("""
    SELECT order_date,
           total_orders,
           ROUND(gross_revenue, 2)         AS gross_revenue,
           ROUND(avg_order_value, 2)        AS avg_order_value,
           unique_customers,
           ROUND(conversion_rate * 100, 1) AS conversion_rate_pct,
           ROUND(fraud_rate * 100, 2)       AS fraud_rate_pct
    FROM curated_daily_kpis
    ORDER BY order_date DESC
    LIMIT 30
""")

kpis['order_date']       = pd.to_datetime(kpis['order_date'])
kpis['gross_revenue']    = kpis['gross_revenue'].astype(float)
kpis['total_orders']     = kpis['total_orders'].astype(int)
kpis['avg_order_value']  = kpis['avg_order_value'].astype(float)
kpis['fraud_rate_pct']   = kpis['fraud_rate_pct'].astype(float)
kpis = kpis.sort_values('order_date')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Daily Gross Revenue (R$)', 'Daily Orders',
                    'Avg Order Value (R$)', 'Fraud Rate (%)'),
)

for (row, col, y_col, color) in [
    (1, 1, 'gross_revenue',   '#2196F3'),
    (1, 2, 'total_orders',    '#4CAF50'),
    (2, 1, 'avg_order_value', '#FF9800'),
    (2, 2, 'fraud_rate_pct',  '#F44336'),
]:
    fig.add_trace(
        go.Scatter(x=kpis['order_date'], y=kpis[y_col],
                   mode='lines+markers', line=dict(color=color, width=2),
                   name=y_col),
        row=row, col=col,
    )

fig.update_layout(title='Ecommerce KPIs — Last 30 Days', height=600, showlegend=False)
fig.show()

## 2. Revenue by Category (Olist)

In [ ]:
cat_rev = run_query("""
    SELECT
        item.category,
        COUNT(DISTINCT o.order_id)                      AS total_orders,
        SUM(item.quantity)                              AS units_sold,
        ROUND(SUM(item.quantity * item.unit_price), 2)  AS gross_revenue,
        ROUND(AVG(item.unit_price), 2)                  AS avg_unit_price,
        ROUND(AVG(item.freight_value), 2)               AS avg_freight
    FROM processed_orders o
    CROSS JOIN UNNEST(items) AS t(item)
    WHERE o.status = 'confirmed'
    GROUP BY 1
    ORDER BY gross_revenue DESC
    LIMIT 15
""")
cat_rev['gross_revenue'] = cat_rev['gross_revenue'].astype(float)
cat_rev['avg_freight']   = cat_rev['avg_freight'].astype(float)
cat_rev['units_sold']    = cat_rev['units_sold'].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Revenue by category
axes[0].barh(cat_rev['category'][::-1], cat_rev['gross_revenue'][::-1], color='steelblue')
axes[0].set_xlabel('Revenue (R$)')
axes[0].set_title('Top 15 Categories by Revenue')

# Avg freight by category
axes[1].barh(cat_rev['category'][::-1], cat_rev['avg_freight'][::-1], color='coral')
axes[1].set_xlabel('Avg Freight (R$)')
axes[1].set_title('Average Freight Cost by Category')

plt.suptitle('Olist Category Performance', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Revenue by Brazilian State

In [ ]:
state_rev = run_query("""
    SELECT
        shipping_state,
        total_orders,
        ROUND(gross_revenue, 2)    AS gross_revenue,
        ROUND(avg_order_value, 2)  AS avg_order_value,
        unique_customers
    FROM curated_state_revenue
    ORDER BY gross_revenue DESC
""")
state_rev['gross_revenue']   = state_rev['gross_revenue'].astype(float)
state_rev['avg_order_value'] = state_rev['avg_order_value'].astype(float)
state_rev['total_orders']    = state_rev['total_orders'].astype(int)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Gross Revenue by State (R$)', 'Avg Order Value by State (R$)'),
)

fig.add_trace(go.Bar(
    x=state_rev['shipping_state'], y=state_rev['gross_revenue'],
    marker_color='#2196F3', name='Revenue',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=state_rev['shipping_state'], y=state_rev['avg_order_value'],
    marker_color='#FF9800', name='Avg Order Value',
), row=1, col=2)

fig.update_layout(title='Brazil State-Level E-Commerce Analysis', height=500, showlegend=False)
fig.show()

## 4. Payment Method Breakdown

In [ ]:
payments = run_query("""
    SELECT
        payment_method,
        COUNT(*)                                              AS order_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)   AS share_pct,
        ROUND(AVG(CAST(total AS DOUBLE)), 2)                  AS avg_order_value,
        ROUND(AVG(CAST(fraud_score AS DOUBLE)), 4)            AS avg_fraud_score
    FROM processed_orders
    WHERE status = 'confirmed'
    GROUP BY payment_method
    ORDER BY order_count DESC
""")
payments['order_count']     = payments['order_count'].astype(int)
payments['share_pct']       = payments['share_pct'].astype(float)
payments['avg_order_value'] = payments['avg_order_value'].astype(float)
payments['avg_fraud_score'] = payments['avg_fraud_score'].astype(float)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Market share pie
axes[0].pie(payments['share_pct'], labels=payments['payment_method'],
            autopct='%1.1f%%', startangle=140)
axes[0].set_title('Payment Method Market Share')

# Avg order value by method
axes[1].bar(payments['payment_method'], payments['avg_order_value'], color='steelblue')
axes[1].set_title('Avg Order Value by Payment (R$)')
axes[1].set_xlabel('Payment Method')

# Avg fraud score — note boleto (bank_transfer) should be low
colors = ['crimson' if s > 0.2 else 'steelblue' for s in payments['avg_fraud_score']]
axes[2].bar(payments['payment_method'], payments['avg_fraud_score'], color=colors)
axes[2].axhline(0.2, color='crimson', linestyle='--', alpha=0.5, label='Threshold')
axes[2].set_title('Avg Fraud Score by Payment')
axes[2].set_xlabel('Payment Method')
axes[2].legend()

plt.suptitle('Payment Method Analysis (Olist)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Customer CLV Segments

In [ ]:
clv = run_query("""
    SELECT clv_segment,
           COUNT(*) AS customer_count,
           ROUND(AVG(total_spend), 2)  AS avg_clv,
           ROUND(AVG(order_count), 1)  AS avg_orders
    FROM curated_customer_clv
    GROUP BY clv_segment
    ORDER BY avg_clv DESC
""")
clv['customer_count'] = clv['customer_count'].astype(int)
clv['avg_clv']        = clv['avg_clv'].astype(float)
clv['avg_orders']     = clv['avg_orders'].astype(float)

colors = {'high': '#2196F3', 'medium': '#FF9800', 'low': '#9E9E9E'}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, label) in zip(axes, [
    ('customer_count', 'Customers per Segment'),
    ('avg_clv',        'Avg Lifetime Value (R$)'),
    ('avg_orders',     'Avg Orders per Customer'),
]):
    bar_colors = [colors.get(s, 'grey') for s in clv['clv_segment']]
    ax.bar(clv['clv_segment'], clv[col], color=bar_colors)
    ax.set_title(label)
    ax.set_xlabel('CLV Segment')

plt.suptitle('Customer Lifetime Value Segmentation (Olist thresholds: high>R$500, medium>R$150)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 6. Fraud Detection Analysis

In [ ]:
fraud = run_query("""
    SELECT order_date,
           COUNT(*)                                                             AS total_orders,
           SUM(CASE WHEN CAST(fraud_score AS DOUBLE) >= 0.5 THEN 1 ELSE 0 END) AS flagged,
           ROUND(AVG(CAST(fraud_score AS DOUBLE)), 4)                           AS avg_score,
           ROUND(100.0 * SUM(CASE WHEN CAST(fraud_score AS DOUBLE) >= 0.5
                                  THEN 1 ELSE 0 END) / COUNT(*), 2)            AS fraud_pct
    FROM processed_orders
    GROUP BY order_date
    ORDER BY order_date DESC
    LIMIT 14
""")
fraud['order_date']   = pd.to_datetime(fraud['order_date'])
fraud['fraud_pct']    = fraud['fraud_pct'].astype(float)
fraud['avg_score']    = fraud['avg_score'].astype(float)
fraud['total_orders'] = fraud['total_orders'].astype(int)
fraud = fraud.sort_values('order_date')

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(fraud['order_date'], fraud['total_orders'], alpha=0.4, color='steelblue', label='Total Orders')
ax2.plot(fraud['order_date'], fraud['fraud_pct'], color='crimson', marker='o', lw=2, label='Fraud Rate %')
ax2.axhline(y=fraud['fraud_pct'].mean(), color='crimson', linestyle='--', alpha=0.5, label='Avg fraud rate')

ax1.set_ylabel('Total Orders', color='steelblue')
ax2.set_ylabel('Fraud Rate (%)', color='crimson')
ax1.set_title('Daily Order Volume vs Fraud Rate')
fig.legend(loc='upper right', bbox_to_anchor=(1, 1), bbox_transform=ax1.transAxes)
plt.tight_layout()
plt.show()

## 7. Freight Cost by State

In [ ]:
freight = run_query("""
    SELECT
        shipping_state,
        COUNT(DISTINCT order_id)                               AS orders,
        ROUND(AVG(CAST(total AS DOUBLE)), 2)                   AS avg_order_value,
        ROUND(AVG(item.freight_value), 2)                      AS avg_freight,
        ROUND(100.0 * AVG(item.freight_value) /
              NULLIF(AVG(CAST(total AS DOUBLE)), 0), 1)        AS freight_pct_of_order
    FROM processed_orders
    CROSS JOIN UNNEST(items) AS t(item)
    WHERE status = 'confirmed'
      AND shipping_state IS NOT NULL
    GROUP BY shipping_state
    ORDER BY avg_freight DESC
""")
freight['avg_freight']          = freight['avg_freight'].astype(float)
freight['avg_order_value']      = freight['avg_order_value'].astype(float)
freight['freight_pct_of_order'] = freight['freight_pct_of_order'].astype(float)

fig = px.scatter(
    freight,
    x='avg_order_value',
    y='avg_freight',
    size='freight_pct_of_order',
    text='shipping_state',
    color='freight_pct_of_order',
    color_continuous_scale='Reds',
    title='Avg Freight vs Avg Order Value by Brazilian State',
    labels={
        'avg_order_value': 'Avg Order Value (R$)',
        'avg_freight': 'Avg Freight (R$)',
        'freight_pct_of_order': 'Freight % of Order',
    },
)
fig.update_traces(textposition='top center')
fig.update_layout(height=550)
fig.show()

## 8. Clickstream Funnel

In [ ]:
funnel = run_query("""
    SELECT action,
           COUNT(*)                    AS event_count,
           COUNT(DISTINCT session_id)  AS sessions,
           ROUND(AVG(duration_ms)/1000.0, 1) AS avg_sec
    FROM processed_clickstream
    GROUP BY action
    ORDER BY event_count DESC
""")
funnel['event_count'] = funnel['event_count'].astype(int)

fig = go.Figure(go.Funnel(
    y=funnel['action'],
    x=funnel['event_count'],
    textinfo='value+percent initial',
    marker=dict(color=['#2196F3', '#42A5F5', '#64B5F6', '#90CAF9', '#BBDEFB', '#E3F2FD'][:len(funnel)]),
))
fig.update_layout(title='Clickstream Action Funnel', height=450)
fig.show()

## 9. Pipeline Health — CloudWatch Metrics

In [ ]:
from datetime import datetime, timedelta, timezone

cw = boto3.client('cloudwatch', region_name=REGION)

def get_metric(namespace, metric_name, dimensions, stat='Sum', period=300, hours=24):
    now = datetime.now(timezone.utc)
    resp = cw.get_metric_statistics(
        Namespace=namespace,
        MetricName=metric_name,
        Dimensions=dimensions,
        StartTime=now - timedelta(hours=hours),
        EndTime=now,
        Period=period,
        Statistics=[stat],
    )
    pts = sorted(resp['Datapoints'], key=lambda x: x['Timestamp'])
    return pd.DataFrame({
        'timestamp': [p['Timestamp'] for p in pts],
        'value': [p[stat] for p in pts],
    })

fn_names = [
    'ecom-order-processor',
    'ecom-stream-enricher',
    'ecom-fraud-detector',
    'ecom-inventory-alerter',
]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()

for ax, fn in zip(axes, fn_names):
    dims = [{'Name': 'FunctionName', 'Value': fn}]
    inv  = get_metric('AWS/Lambda', 'Invocations', dims)
    errs = get_metric('AWS/Lambda', 'Errors', dims)

    if not inv.empty:
        ax.bar(inv['timestamp'], inv['value'], width=0.003, color='steelblue', alpha=0.7, label='Invocations')
    if not errs.empty:
        ax.bar(errs['timestamp'], errs['value'], width=0.003, color='crimson', alpha=0.9, label='Errors')

    ax.set_title(fn.replace('ecom-', ''))
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('Lambda Invocations vs Errors — Last 24h', fontsize=14)
plt.tight_layout()
plt.show()